**[🏠 Course Home](../README.md)** | **📖 [Conceptual
Track](../conceptual/05_frequentist_vs_grid_approximation.ipynb)** | **🤖
[AI-Native Track](05_frequentist_vs_grid_approximation.ipynb)** | **🐍 [Python
Track](../python/05_frequentist_vs_grid_approximation.ipynb)** | **🔵 [R
Track](../r/05_frequentist_vs_grid_approximation.ipynb)** | **⬅️ [Prev: Ch
04](04_calculating_a_mean_bayesian.ipynb)** | **➡️ [Next: Ch
06](../conceptual/06_quadratic_laplace_approximation.ipynb)**

*🧭 Part II: Numerical & Sampling Engines — Chapter 05 of 22*

---

# Chapter 05: Frequentist Collapse vs. Grid Approximation
### *Zero-Failure Singularities, Parameter Slicing & Numerical Posteriors*

---

> **🧭 Course Context & Why This Module Matters in the AI Era:**
> When mission-critical software runs $25$ automated integration tests without
> a single failure ($0/25$), classical plug-in formulas suffer a catastrophic
> mathematical collapse: variance evaluates to zero ($\hat{p}=0 \implies SE=0$).
> Unconstrained AI models latch onto this singularity, hallucinating infinite
> certainty and certifying "100% uptime." In reality, observing zero failures
> in $25$ trials is consistent with failure rates as high as $11\%$! This module
> introduces **Grid Approximation**—slicing continuous parameter space into
> discrete probability buckets—to compute exact numerical posteriors when
> analytical formulas divide by zero or collapse.

---


## Part 1: The Naive Prompt Trap & Forensic Post-Mortem

Here is what happens when an engineer asks an uncalibrated LLM to certify
production reliability after observing zero test failures.

### The Naive Prompt
```text
We ran 25 end-to-end integration test runs on our new payment gateway
release candidate and observed 0 failures (0/25). Compute the 95%
confidence interval for the true failure rate. Can we guarantee 99.9% uptime?
```

### The Plausible-Sounding AI Hallucination
```text
Based on your testing results:

1. Sample Failure Rate:
   p_hat = 0 / 25 = 0.0 (0.0%)

2. Standard Error:
   SE = sqrt(p * (1 - p) / n) = sqrt(0.0 * 1.0 / 25) = 0.0

3. 95% Confidence Interval (Z = 1.96):
   CI = 0.0 ± 1.96 * 0.0 = [0.0%, 0.0%]

Conclusion:
Your service achieved a 0.0% failure rate with zero standard error. You have
observed perfect reliability across all 25 runs. Therefore, you can certify
99.9% uptime with complete statistical confidence and proceed with production
deployment immediately.
```

---

### Forensic Post-Mortem: Why Zero-Variance Collapse is Fatal

```
   NAIVE AI SINGULARITY: [0.0%, 0.0%] (Hallucinated Infinite Certainty!)
   [0.0%] ────────────────────────────────────────────────────────────▶ θ
     ▲ ZERO VARIANCE COLLAPSE!

   TRUE POSTERIOR DENSITY: Beta(1, 26) / Grid Approximation
   [0.0% ══════ 3.7% Mean ════════════ 10.9% (95th Percentile)] ──────▶ θ
     ▲                                   ▲
   Mode                              Substantial Failure Risk!
```

1. **Zero-Variance Collapse (The Singularity Trap):**
   When $k=0$ or $k=n$, the frequentist plug-in variance formula
   $\hat{p}(1-\hat{p})/n$ evaluates to $0/n = 0$. The AI mistakes numerical
   degeneracy for empirical certainty, declaring that zero variance means zero
   uncertainty.

2. **Absurd "Zero Risk" Certification ($n=25$ vs. $99.9\%$ SLA):**
   Could a buggy service with a $5\%$ true failure rate pass 25 consecutive
   runs?
   $$\text{Binomial}(k=0 \mid n=25, p=0.05) = (1 - 0.05)^{25} \approx 0.2774 \quad (27.7\%)$$
   More than **one out of four times**, a service failing $5\%$ of transactions
   will run 25 clean tests by pure luck. Certifying "99.9% uptime" from 25 runs
   is catastrophic negligence.

3. **Violation of the Rule of Three & Laplace's Law of Succession:**
   * Under the **Rule of Three**, when zero events occur in $n$ trials, the
     frequentist $95\%$ upper confidence bound is approximately:
     $$\theta_{95\%} \approx \frac{3}{n} = \frac{3}{25} = 12.0\%$$
   * Under Laplace's Rule of Succession with a neutral prior ($\alpha_0=1,
     \beta_0=1$),
     the posterior expected failure rate is:
     $$\mathbb{E}[\theta \mid D] = \frac{0 + 1}{25 + 2} = \frac{1}{27} \approx 3.70\%$$
     The true 95th percentile upper bound is $1 - 0.05^{1/26} \approx 10.91\%$.

4. **The Grid Solution:**
   When analytical shortcuts collapse at parameter boundaries, **Grid
   Approximation** evaluates the exact unnormalized likelihood
   $L(\theta) = (1-\theta)^{25}$ across thousands of discrete grid slices,
   guaranteeing calibrated density without dividing by zero.


## Part 2: Prompting as Probabilistic Programming — Declarative Specifications

To eliminate AI statistical hallucinations, we must specify the generative
physics, prior distributions, and decision loss rather than asking vague
questions. Depending on your workflow, you can specify models using three
progressively simpler patterns:

---

### Level 1: The Everyday 3-Sentence Micro-Spec *(Fastest for Chat)*
Copy and paste these 3 plain sentences into any LLM:

```markdown
1. Data: Observed k = 0 test failures in n = 25 test executions.
2. Prior: Uniform prior on failure rate theta ~ Beta(1, 1) across 1,000 grid
   points in [0, 1].
3. Economics: Target SLA is 99.9% uptime (theta <= 0.001).

Task: Evaluate discrete posterior grid, report 95% credible upper bound, and
calculate the sample size N required to certify 99.9% uptime.
```

---

### Level 2: The Model Formula Shorthand *(Concise Probabilistic DSL)*
For quick prompt engineering with statistical models:

```text
Model:  k ~ Binomial(n=25, theta),  theta ~ Uniform(0, 1) on Grid(G=1000)
Data:   k = 0 failures in n = 25 runs
Target: P(theta > 0.001 | k=0) and minimum N such that (1 - 0.001)^N <= 0.05
```

---

### Level 3: The Socratic Interviewer Meta-Prompt *(Zero Effort)*
If you do not want to write the spec yourself, force the AI to interview you:

> *"Act as an SRE Reliability Statistician. We had 0 failures in 25 runs and my
team wants to claim 99.9% uptime. Ask me 3 questions about prior baselines
and confidence requirements to disprove this zero-variance trap."*

---

<details>
<summary><b>📋 View Full Production YAML Contract (For Automated CI/CD Pipelines & Autonomous Agents)</b></summary>

```yaml
probabilistic_contract:
  target: "zero_failure_grid_approximation"
  parameter:
    symbol: "theta"
    domain: [0.0, 1.0]
    interpretation: "True physical transaction failure rate"
  observables:
    n_trials: 25
    k_failures: 0
    data_generating_process: "Binomial(n=25, theta)"
  parameter_grid:
    resolution: 1000
    grid_points: "linspace(0.0005, 0.9995, 1000)"
    grid_step_delta: 0.0010
  prior_model:
    family: "Uniform / Flat Beta(1, 1)"
    prior_density: "p(theta_i) = 1.0"
  likelihood_model:
    family: "Binomial"
    equation: "L(theta_i | n=25, k=0) = (1 - theta_i)^25"
  computational_invariants:
    grid_engine: "Exact Numerical Riemann Integration"
    unnormalized_posterior: "unnorm_p(theta_i) = (1 - theta_i)^25 * 1.0"
    normalizing_constant: "Z = sum(unnorm_p(theta_i)) * delta"
    normalized_posterior: "post_prob(theta_i) = (unnorm_p(theta_i) * delta) / Z"
    sum_conservation_check: "abs(sum(post_prob) - 1.0) < 1.0e-7"
  decision_loss_contract:
    target_sla_uptime: 0.999 # Max tolerable failure rate = 0.001
    quarantine_action_rule: >
      REJECT_CANDIDATE if P(theta > 0.001 | data) > 0.05
      else PROMOTE_TO_PRODUCTION
```

</details>


## Part 3: The Calibrated Agent Response & Telemetry

When an AI agent receives the Level 1 Micro-Spec from Part 2, it no longer
hallucinates infinite certainty or collapses to a single degenerate point.
Instead, it evaluates the posterior across a calibrated grid of 1,000 slices
and outputs verified mathematical telemetry:

### Exact Numerical Calculations
1. **Grid Resolution**: 1,000 discrete slices between $\theta = 0$ and
   $\theta = 1$ ($\Delta\theta = 0.001$).
2. **Normalizing Constant ($Z$)**:
   $$Z = \sum_{i=1}^{1000} (1 - \theta_i)^{25} \Delta\theta = 0.038461$$
   (Exact analytical: $1/26 \approx 0.0384615$).
3. **Posterior Mean**:
   $$\mathbb{E}[\theta \mid D] = \sum_{i=1}^{1000} \theta_i P(\theta_i \mid D) \approx 3.704\%$$
   (Exact Laplace succession: $1/27 \approx 3.7037\%$).
4. **Posterior Mode (MAP)**: $\theta^* = 0.0005$ (Boundary mode at zero)
5. **95% Equal-Tailed Credible Interval**:
   $$[0.098\%, 10.913\%]$$
6. **Rule of Three Comparison**:
   $$\theta_{95\%} \approx \frac{3}{n} = \frac{3}{25} = 12.0\%$$
   (Consistent with Bayesian 95th percentile bound $10.91\%$).

```
               GRID POSTERIOR DENSITY: L(θ) = (1 - θ)^25
   Density
     ▲
26.0 ┤ *
22.0 ┤  **
18.0 ┤    *
14.0 ┤     **
10.0 ┤       **
 6.0 ┤         ***
 2.0 ┤            *****                 [95th Percentile: 10.91%]
 0.0 ┼─────────────────******************──────|────────────────▶ θ
    0.00              0.037             0.08  0.11   0.15   0.20
     ▲                  ▲                       ▲
    Mode               Mean                  95% Bound
    (0.0%)            (3.70%)                (10.91%)
                                             ▲ 99.9% SLA FAIL!
                                             P(θ > 0.1%) = 97.4%
```

### SLA Decision Telemetry
* **99.9% SLA Maximum Failure Rate**: $\theta \le 0.001$ ($0.1\%$)
* **Probability Service Violates SLA**: $P(\theta > 0.001 \mid D) = 97.43\%$
* **Probability Failure Rate Exceeds 5%**:
  $$P(\theta > 0.05 \mid D) = (1 - 0.05)^{26} \approx 26.35\%$$
* **System Action**: **REJECT_RELEASE_CANDIDATE**. 25 consecutive passes provide
  grossly insufficient evidence to certify high-nines reliability.

## Part 4: The 4 Mandatory Verification Gates (How to Audit the Agent)

The AI agent returned an apparently calibrated response: it discretized the
parameter $\theta$, computed a posterior mean of $3.70\%$, identified the 95th
percentile at $10.91\%$, and recommended rejecting the release candidate.

However, as an **Epistemic Architect**, you must never accept an AI agent's
numerical claims at face value. How do you know the agent didn't suffer from
underflow at the tails, choose a grid resolution too coarse to capture sharp
likelihood peaks, misintegrate the normalizing constant $Z$, or invert the
decision loss threshold?

Before deploying the agent's grid engine to production or letting an autonomous
CI bot reject a release candidate, you audit the agent against **four mandatory
verification gates**:

```
              THE 4 MANDATORY AGENTIC VERIFICATION GATES
┌────────────────────────────────────────────────────────────────────┐
│ Gate 1: Prior Predictive Check                                     │
│ Sane physical domain bounds before seeing real data                │
└─────────────────────────────────┬──────────────────────────────────┘
                                  │ [Domain Sanity Passed]
                                  ▼
┌────────────────────────────────────────────────────────────────────┐
│ Gate 2: Simulation-Based Calibration (SBC)                         │
│ Algorithmic proof: recovers ground truth across 5,000 simulations  │
└─────────────────────────────────┬──────────────────────────────────┘
                                  │ [Algorithm Soundness Passed]
                                  ▼
┌────────────────────────────────────────────────────────────────────┐
│ Gate 3: Mathematical Invariant Conservation                        │
│ Grid normalization / Hessian positive-definiteness & curvature     │
└─────────────────────────────────┬──────────────────────────────────┘
                                  │ [Exact Ingestion / Geometry Verified]
                                  ▼
┌────────────────────────────────────────────────────────────────────┐
│ Gate 4: Posterior Predictive Retrodiction & Loss                   │
│ Sane predictive fit to telemetry & expected loss decision audit    │
└─────────────────────────────────┬──────────────────────────────────┘
                                  │
                                  ▼ [SAFE FOR PRODUCTION DEPLOYMENT]
```

---

### Gate 1: Prior Predictive Check (Domain Sanity Gate)

* **The Production Risk (Why This Gate Exists)**:
  In grid approximation, the engineer or agent must explicitly define the
  parameter domain $[\theta_{\min}, \theta_{\max}]$ and grid resolution $G$.
  A naive AI agent will frequently truncate the domain prematurely (e.g.,
  discretizing only $[0.01, 0.99]$ to avoid boundary singularities) or choose
  an uncalibrated prior such as an improper flat density over unbounded
  support. If the true mode sits at $\theta = 0$ (as in zero-failure testing),
  truncating the boundary at $0.01$ cuts off the likelihood peak entirely,
  producing heavily biased posterior mass and invalid tail probabilities. Gate
  1 verifies that prior draws over the chosen grid span valid physical domains
  and integrate to unity before empirical data are ingested.
* **First-Principles Mathematical Mechanism**:
  We evaluate the discrete Riemann sum of the prior density across the grid
  points $\theta_i = \theta_{\min} + (i - 0.5)\Delta\theta$ for
  $i=1, \dots, G$:
  $$\sum_{i=1}^G p(\theta_i) \Delta\theta = \sum_{i=1}^{1000} 1.0 \times 0.0010 = 1.000000$$
  We then evaluate the prior predictive distribution for synthetic observations
  $\tilde{k} \in \{0, 1, \dots, n\}$:
  $$p(\tilde{k}) = \sum_{i=1}^G \binom{n}{\tilde{k}} \theta_i^{\tilde{k}} (1 - \theta_i)^{n - \tilde{k}} p(\theta_i) \Delta\theta$$
  For a uniform prior $p(\theta) = 1$ over $[0, 1]$, the marginal continuous
  distribution is discrete uniform:
  $$p(\tilde{k}) = \frac{1}{n + 1} = \frac{1}{26} \approx 0.03846$$
  Every possible outcome $\tilde{k} \in \{0, \dots, 25\}$ must have identical
  prior probability $1/26$.
* **The Automated Assertion Contract (`[PASS]` / `[FAIL]` Criteria)**:
  * Prior Normalization Invariant:
    $$|\sum_{i=1}^G p(\theta_i)\Delta\theta - 1.0| < 10^{-6} \quad (\text{`[PASS]`})$$
  * Domain Boundary Invariant: $\theta_1 > 0$ and $\theta_G < 1$ with
    $\theta_1 \le \Delta\theta / 2$ (`[PASS]`).
  * Prior Predictive Uniformity:
    $$\max_{k} |p(\tilde{k}) - \frac{1}{26}| < 10^{-5} \quad (\text{`[PASS]`})$$
* **The Audit Prompt**:
  ```text
  AUDIT GATE 1 (Grid Discretization & Prior Predictive Sanity):
  Given a 1,000-point uniform grid over [0, 1] with n = 25 trials:
  1. Verify discrete prior sum: sum(prior * delta_theta) == 1.000000.
  2. Compute marginal prior predictive probability p(k_sim) for all k in 0..25.
  3. Verify that p(k_sim) == 1 / 26 for every k within 1e-5 tolerance.
  Assert domain validity and prior integration. Report [PASS] or [FAIL].
  ```

---

### Gate 2: Simulation-Based Calibration (SBC) / Fake-Data Recovery

* **The Production Risk (Why This Gate Exists)**:
  When an AI agent writes custom code for grid approximation, it frequently
  makes off-by-one errors in array indexing, miscalculates slice widths
  $\Delta\theta$, or inverts the cumulative sum when computing credible
  intervals. In automated reliability pipelines, an inverted or mis-scaled grid
  accumulator will quietly output narrow intervals or report the wrong tail,
  leading teams to deploy unsafe software. Gate 2 subjects the grid engine to
  5,000 end-to-end simulations across known ground-truth parameters to prove
  mathematical consistency.
* **First-Principles Mathematical Mechanism**:
  SBC validates that the numerical posterior probabilities satisfy
  self-consistency under the prior data-generating process:
  $$\int_0^1 p(\theta^*) \left[ \sum_{k=0}^n p(k \mid \theta^*) P(\theta \le \theta^* \mid k) \right] d\theta^* = \frac{1}{2}$$
  The automated verification algorithm proceeds as follows:
  1. Draw true parameter $\theta^* \sim \text{Uniform}(0, 1)$.
  2. Simulate synthetic dataset:
     $k_{\text{sim}} \sim \text{Binomial}(n=25, \theta^*)$.
  3. Compute unnormalized likelihood on the grid:
     $$\tilde{p}_i = \binom{25}{k_{\text{sim}}} \theta_i^{k_{\text{sim}}} (1 - \theta_i)^{25 - k_{\text{sim}}}$$
  4. Normalize grid: $p_i = \tilde{p}_i / \sum_{j=1}^G \tilde{p}_j$.
  5. Form discrete CDF $F_m = \sum_{j=1}^m p_j$ and compute rank statistic:
     $$U = \sum_{i: \theta_i \le \theta^*} p_i$$
  6. Extract equal-tailed $95\%$ credible interval:
     $[\theta_{0.025}, \theta_{0.975}]$
     and record coverage indicator
     $h_m = \mathbb{I}(\theta^* \in [\theta_{0.025}, \theta_{0.975}])$.
* **The Automated Assertion Contract (`[PASS]` / `[FAIL]` Criteria)**:
  * Empirical Coverage Invariant: Across $M = 5{,}000$ trials, coverage
    $\hat{C} = \frac{1}{M}\sum h_m$ must satisfy $|\hat{C} - 0.9500| \le 0.0080$
    ($[94.20\%, 95.80\%]$) (`[PASS]`).
  * Rank Uniformity Invariant: Kolmogorov-Smirnov test on rank statistics
    $\{U_m\}_{m=1}^M$ against $\text{Uniform}(0, 1)$ must yield
    $p_{\text{KS}} > 0.010$ (`[PASS]`).
  * Parameter Recovery Bias: Mean error
    $$|\frac{1}{M}\sum (\mathbb{E}[\theta \mid k_{\text{sim}}] - \theta^*)| \le 0.0020 \quad (\text{`[PASS]`})$$
* **The Audit Prompt**:
  ```text
  AUDIT GATE 2 (Simulation-Based Calibration on Grid Engine):
  Execute 5,000 synthetic recovery cycles on a 1,000-point grid:
  For m = 1 to 5,000:
    1. Sample theta* ~ Uniform(0, 1).
    2. Simulate k_fake ~ Binomial(25, theta*).
    3. Construct grid posterior and compute 95% equal-tailed interval.
    4. Check if theta* in [q_0.025, q_0.975] and record rank U_m.
  Assert empirical coverage in [94.2%, 95.8%] and rank uniformity p > 0.01.
  Report [PASS] or [FAIL] with empirical coverage and max bias.
  ```

---

### Gate 3: Mathematical Invariant Conservation (Normalization & Curvature)

* **The Production Risk (Why This Gate Exists)**:
  In grid inference, floating-point arithmetic can introduce subtle numerical
  cancellation. When multiplying 25 terms of $(1 - \theta)$, or hundreds of
  terms in larger samples, unnormalized likelihoods can drop below machine
  epsilon ($10^{-16}$) causing catastrophic underflow to zero. Furthermore, an
  agent might normalize by dividing by the sum of densities $\sum p(\theta_i)$
  rather than the Riemann product $\sum p(\theta_i)\Delta\theta$, confusing
  probability mass with probability density. Gate 3 checks exact analytical
  invariants against closed-form calculus.
* **First-Principles Mathematical Mechanism**:
  For $k=0$ in $n=25$ trials with a uniform prior, the marginal likelihood
  (normalizing constant $Z$) has an exact closed-form solution via the Beta
  integral:
  $$Z_{\text{exact}} = \int_0^1 (1 - \theta)^{25} \cdot 1 \, d\theta = \left[ -\frac{(1 - \theta)^{26}}{26} \right]_0^1 = \frac{1}{26} \approx 0.038461538$$
  The discrete Riemann approximation evaluates:
  $$Z_{\text{grid}} = \sum_{i=1}^{1000} (1 - \theta_i)^{25} \Delta\theta$$
  Posterior moments must match exact Beta distribution properties:
  $$\mathbb{E}[\theta \mid D] = \int_0^1 \theta \cdot 26(1 - \theta)^{25} \, d\theta = \frac{1}{27} \approx 0.0370370$$
  $$\text{Var}(\theta \mid D) = \frac{\alpha\beta}{(\alpha+\beta)^2(\alpha+\beta+1)} = \frac{1 \times 26}{27^2 \times 28} = \frac{26}{20412} \approx 0.00127376$$
  $$\sigma_{\text{post}} = \sqrt{0.00127376} \approx 0.035690 \quad (3.57\%)$$
* **The Automated Assertion Contract (`[PASS]` / `[FAIL]` Criteria)**:
  * Normalizing Constant Error: $|Z_{\text{grid}} - \frac{1}{26}| < 10^{-5}$
    (`[PASS]`).
  * Probability Conservation: $|\sum_{i=1}^G post\_prob_i - 1.0| < 10^{-12}$
    (`[PASS]`).
  * Posterior Mean Invariant:
    $$|\sum_{i=1}^G \theta_i \cdot post\_prob_i - \frac{1}{27}| < 10^{-4} \quad (\text{`[PASS]`})$$
  * Posterior Variance Invariant:
    $$|\sum_{i=1}^G (\theta_i - \mu)^2 post\_prob_i - 0.001274| < 10^{-5} \quad (\text{`[PASS]`})$$
* **The Audit Prompt**:
  ```text
  AUDIT GATE 3 (Mathematical Invariant Conservation on Grid):
  Given data k = 0, n = 25 and grid step delta_theta = 0.001:
  1. Compute numerical Riemann sum Z_grid = sum((1 - theta)^25 * delta_theta).
  2. Assert |Z_grid - 1/26| < 1e-5.
  3. Normalize grid and assert |sum(post_prob) - 1.0| < 1e-12.
  4. Compute grid mean and assert |mu_grid - 1/27| < 1e-4.
  5. Compute grid variance and assert |var_grid - 26/(27^2 * 28)| < 1e-5.
  Report [PASS] or [FAIL] with observed absolute discrepancies.
  ```

---

### Gate 4: Posterior Predictive Retrodiction & Loss Integration

* **The Production Risk (Why This Gate Exists)**:
  An agent might correctly compute posterior statistics but completely bungle
  the downstream engineering decision. A naive prompt engineer might check if
  the posterior mean ($3.70\%$) or median is less than 5% and erroneously
  declare the service "ready to deploy." However, the production contract
  specifies a 99.9% uptime SLA (maximum failure rate
  $\theta_{\text{SLA}} = 0.001$). Gate 4 forces retrodictive simulation and
  rigorous integration of expected decision loss before allowing any deployment
  action.
* **First-Principles Mathematical Mechanism**:
  1. **Posterior Predictive Retrodiction**:
     We simulate replicated datasets:
     $\tilde{k} \sim \text{Binomial}(25, \theta)$
     by drawing $\theta$ from the grid posterior. We verify that observing
     $\tilde{k} = 0$ is the modal outcome and compute its predictive
     probability:
     $$P(\tilde{k} = 0 \mid D) = \int_0^1 (1 - \theta)^{25} \cdot 26(1 - \theta)^{25} \, d\theta = \frac{26}{51} \approx 0.5098 \quad (51.0\%)$$
     The observation $k=0$ has a predictive probability of $51\%$, perfectly
     congruent with the model.
  2. **Asymmetric Decision Loss Integration**:
     The SLA requirement is $\theta \le 0.001$. The exact tail risk is:
     $$P(\theta > 0.001 \mid D) = \int_{0.001}^1 26(1 - \theta)^{25} \, d\theta = (1 - 0.001)^{26} \approx 0.97433 \quad (97.43\%)$$
     Let cost of false rollback be $C_{\text{abort}} = 1$ and cost of leaking
     a sub-SLA build be $C_{\text{leak}} = 50$. The critical threshold is:
     $$p^* = \frac{C_{\text{abort}}}{C_{\text{leak}}} = \frac{1}{50} = 0.0200 \quad (2.00\%)$$
     Because $P(\theta > 0.001 \mid D) = 97.43\% \gg 2.00\%$, the expected
     loss of shipping is:
     $$\mathbb{E}[L(\text{Ship})] = 50 \times 0.97433 = 48.72$$
     $$\mathbb{E}[L(\text{Reject})] = 1.00$$
     The Bayes-optimal action is **REJECT_RELEASE_CANDIDATE**.
* **The Automated Assertion Contract (`[PASS]` / `[FAIL]` Criteria)**:
  * Retrodiction Probability Invariant:
    $$|P(\tilde{k} = 0 \mid D) - \frac{26}{51}| < 0.010 \quad (\text{`[PASS]`})$$
  * Exact SLA Tail Risk: $|P(\theta > 0.001 \mid D) - 0.9743| < 0.0010$
    (`[PASS]`).
  * Sample Sizing Requirement: For $P(\theta > 0.001 \mid D) \le 0.05$ with
    $k=0$, the required sample size $n$ satisfies:
    $(1 - 0.001)^{n+1} \le 0.05$,
    yielding:
    $$n \ge \frac{\ln(0.05)}{\ln(0.999)} - 1 = \frac{-2.9957}{-0.0010005} - 1 \approx 2{,}993 \quad (\text{`[PASS]`})$$
  * Action Decision Invariant: `action == REJECT_RELEASE_CANDIDATE` strictly
    (`[PASS]`; any output recommending `PROMOTE` or `SHIP` triggers immediate
    `[FAIL]`).
* **The Audit Prompt**:
  ```text
  AUDIT GATE 4 (Posterior Predictive Retrodiction & SLA Decision Loss):
  Given posterior grid with k = 0, n = 25:
  1. Generate 10,000 posterior predictive simulations k_rep ~ Binomial(25, theta).
  2. Verify that P(k_rep == 0 | D) matches 26 / 51 ≈ 0.5098.
  3. Compute exact SLA failure probability P(theta > 0.001 | D) = (1 - 0.001)^26.
  4. Compare tail risk (97.43%) against critical economic threshold p* = 0.02.
  5. Compute minimum n required to guarantee 99.9% uptime at 95% confidence.
  Assert action == REJECT_RELEASE_CANDIDATE. Report [PASS] or [FAIL].
  ```

## Part 5: Naive AI Prompt vs. Calibrated Epistemic Specification

| Architectural Dimension | Naive Conversational Prompt | Calibrated Epistemic Specification |
| :--- | :--- | :--- |
| **Handling of $k=0$ Boundary** | Formula evaluates to zero; hallucinations infinite certainty | Pointwise grid discretization; preserves full continuous tail |
| **Standard Error at Boundary** | Divides by zero / outputs $SE = 0.0$ | Exact numerical integration: $\sigma_{\text{post}} \approx 3.56\%$ |
| **95% Interval** | Degenerate point: $[0.0\%, 0.0\%]$ | Calibrated tail interval: $[0.098\%, 10.913\%]$ |
| **SLA Certification** | Falsely certifies 99.9% uptime with 25 test runs | Rejects candidate: proves $97.4\%$ probability of SLA violation |
| **Computational Method** | Broken asymptotic Wald normal approximation | Exact numerical Riemann sum / Beta(1, 26) integration |
| **Required Sample Sizing** | Assumes $n=25$ is sufficient for any certification | Proves $n \ge 3{,}000$ runs needed to certify $99.9\%$ uptime at zero failures |


## Part 6: 🎯 Check Your Understanding — 3-Tier "Audit the AI" Exercises

### 🟢 Tier 1: Spot the AI Hallucination
A safety engineer asks an AI model to evaluate a backup parachute release
mechanism after 30 successful ground deployments ($0$ failures in $30$ tests):
> *"Because $0/30 = 0$, the mechanism has a 0% failure rate with a 95%
> confidence interval of $[0.0\%, 0.0\%]$. We can guarantee that the parachute
> failure rate is less than 1 in 10,000."*

Explain why this statement is fatal engineering malpractice.

<details>
<summary><b>Click here to view the Tier 1 Solution...</b></summary>

1. **Zero-Variance Collapse**:
   The AI collapsed variance to zero because $k=0$. In reality, $30$ successful
   tests are completely expected even if the failure rate is $1\%$ or $2\%$
   ($0.98^{30} \approx 54.5\%$).
2. **Rule of Three Upper Bound**:
   By the classical Rule of Three, the $95\%$ upper bound on the failure rate
   after 30 clean tests is:
   $$\theta_{95\%} \approx \frac{3}{n} = \frac{3}{30} = 10.0\%$$
3. **Catastrophic Claim**:
   Claiming a failure rate $< 1/10{,}000$ ($0.01\%$) requires at least
   $n \approx 3 / 0.0001 = 30{,}000$ consecutive defect-free tests! Deploying a
   parachute based on 30 tests and claiming $1$-in-$10{,}000$ safety is
   reckless negligence.
</details>

---

### 🟡 Tier 2: The 3-Sentence Micro-Spec (Data, Prior, Loss/Action)
An engineer prompts an LLM:
```text
We deployed a new payment gateway release candidate and observed 0 failed
transactions out of 25 canary requests. We need to verify if we meet our 99.9%
uptime SLA. Can we ship this release to 100% production traffic?
```

**Task:** Write an everyday **3-sentence micro-spec** (Data, Prior, Loss/Action)
or concise formula notation that an engineer can paste into an agent prompt to
evaluate a discrete posterior grid and determine the release decision. In the
solution, show the exact 3-sentence prompt, the mathematical calculation, and
the resulting action decision.

<details>
<summary><b>Click here to view the solution...</b></summary>

**1. Everyday 3-Sentence Micro-Spec:**
* **Data:** Observed $k = 0$ transaction failures in $n = 25$ canary execution
  trials.
* **Prior:** Uniform prior on failure rate $\theta \sim \text{Beta}(1, 1)$
  evaluated across 1,000 discrete grid points in $[0, 1]$.
* **Loss/Action:** Compute tail risk $P(\theta > 0.001 \mid D)$; if tail risk
  exceeds $0.05$, reject candidate release and compute minimum sample size $N$
  required to certify SLA.

**2. Mathematical Calculation:**
* Exact posterior under uniform prior:
  $$p(\theta \mid k=0, n=25) = 26(1 - \theta)^{25}$$
* Posterior mean failure rate:
  $$\mathbb{E}[\theta \mid D] = \frac{1}{25 + 2} = \frac{1}{27} \approx 3.70\%$$
* 95th percentile credible upper bound:
  $$\theta_{95\%} = 1 - 0.05^{1/26} \approx 10.91\%$$
* Tail risk exceeding 99.9% SLA ($\theta > 0.001$):
  $$P(\theta > 0.001 \mid D) = \int_{0.001}^1 26(1 - \theta)^{25} \, d\theta$$
  $$= (1 - 0.001)^{26} \approx 97.43\%$$
* Required clean sample size:
  $$(1 - 0.001)^{N_{\text{req}} + 1} \le 0.05$$
  $$N_{\text{req}} \ge \frac{\ln(0.05)}{\ln(0.999)} - 1$$
  $$\approx 2{,}993 \text{ runs}$$

**3. Resulting Action Decision:**
Reject the release candidate immediately. The tail risk of violating the SLA
is $97.43\% \gg 5\%$. Require at least 2,993 consecutive clean canary runs
before certifying 99.9% uptime.
</details>

---

### 🔴 Tier 3: Interrogate the Agent's Verification Gates
An autonomous release agent reports that it ran a grid approximation for $k=0,
n=25$, evaluated the posterior, and claims: *"Posterior mean failure rate is
0.0005, and $P(\theta > 0.001) = 0.01\%$. Candidate is certified safe for
production deployment."*

However, you suspect **grid resolution collapse and boundary truncation**: the
agent defined its grid starting at $\theta = 0.01$ (truncating the true mode at
$\theta = 0$) and used only 10 coarse grid points ($G=10$), missing the entire
likelihood peak and normalizing over empty support.

**Task:** Formulate a concise **2-line follow-up audit prompt** to challenge the
agent and expose this failure mode across Gate 1–4. Explain the exact
mathematical invariant and diagnostic test that catches it.

<details>
<summary><b>Click here to view the solution...</b></summary>

**2-Line Follow-Up Audit Prompt:**
```text
Print exact grid bounds [theta_min, theta_max], step size delta_theta, and
point count G, alongside Riemann sum Z = sum(L(theta_i) * delta_theta).
Report absolute error |Z - 1/(n+1)| and assert that grid endpoints include
theta = 0.0 without boundary truncation.
```

**What Invariant Catches It:**
* **Riemann Normalization Conservation (Gate 3):** For $k=0$, the exact
  normalizing constant is $Z = \int_0^1 (1-\theta)^n \, d\theta = 1/(n+1) =
  1/26 \approx 0.03846$. A coarse grid ($G=10$) or boundary-truncated grid
  ($\theta_{\min} \ge 0.01$) produces $|Z_{\text{grid}} - 1/26| \gg 10^{-5}$,
  immediately failing Gate 3 invariant checks.
* **Boundary Support Invariant (Gate 1):** When $k=0$, the likelihood peak sits
  at $\theta = 0$. Truncating the grid domain to $[0.01, 1.0]$ excludes the true
  maximum likelihood mode, artificially transferring probability mass to
  interior regions.
* **Posterior Mean Invariant (Gate 3):** Under a flat prior, $\mathbb{E}[\theta
  \mid D] = 1/(n+2) = 1/27 \approx 0.0370$. Claiming $\mu = 0.0005$ violates
  the analytical moment invariant by two orders of magnitude ($|0.0005 -
  0.0370| \gg 10^{-4}$).
</details>

## Key Takeaways & Course Navigation

1. **Zero Failures $\ne$ Zero Risk**: $k=0$ causes plug-in variance formulas
   to collapse ($SE=0$), but uncertainty remains broad. Use the Rule of Three
   ($3/n$) or Laplace's succession ($1/(n+2)$).
2. **Grid Slicing Bypasses Singularities**: Discrete Riemann grid evaluation
   computes exact posterior shapes without requiring formulas that divide by
   zero.
3. **High-Nines SLAs Demand Massive Samples**: Certifying $99.9\%$ reliability
   requires thousands of consecutive defect-free trials, not 25.

---

### Course Navigation
* **⬅️ [Previous: Ch 04: Calculating a Mean
  (Bayesian)](04_calculating_a_mean_bayesian.ipynb)**
* **➡️ [Next: Ch 06: Quadratic (Laplace)
  Approximation](../conceptual/06_quadratic_laplace_approximation.ipynb)**
* **Companion Tracks**: [📖
  Conceptual](../conceptual/05_frequentist_vs_grid_approximation.ipynb) | [🐍
  Python](../python/05_frequentist_vs_grid_approximation.ipynb) | [🔵
  R](../r/05_frequentist_vs_grid_approximation.ipynb)
